### Diff building blocks on one MSFT pair

Step-by-step exploration of `mask()` and `similarity_matrix()` on MSFT's first yoy pair. The final versions (with period-word masking and case-insensitive scoring) are in `src/diff.py`.

In [2]:
import pandas as pd

import sys
sys.path.append("..")

from src import diff

pairs = pd.read_csv(diff.PAIRS_PATH)
paragraphs = pd.read_csv(diff.PARAGRAPHS_PATH)
pair = pairs[(pairs.ticker == "MSFT") & (pairs.pair_type == "yoy")].iloc[0]
old, new = diff.load_pair(pair, paragraphs)
len(old), len(new), old[0]

(110,
 119,
 {'para_no': 0,
  'text': 'ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS',
  'section': None,
  'position': 0.0})

In [ ]:
print(old)
print(new)

# old and new are of the form [{'para_no':X, 'text':paragraph_X}, {}, {}...] for each paragraph in the text

In [10]:
# 1. mask(text) -> text(masked): this fxn should return text with all numbers/ %s/ $s replaced with a placeholder <XX>
import re
# 1. Dates: Matches MM/DD/YYYY, YYYY-MM-DD, written months (Oct 25, 2026), etc.
def mask(text):
    date_pattern = (
        r"\b\d{1,4}[./-]\d{1,4}[./-]\d{2,4}\b|"
        r"\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{2,4}\b|"
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+\d{2,4}\b"
    )

    # 2. Currency: Matches $100, £50.25, $1,000.00, etc.
    currency_pattern = r"[$_£¥€]\s*\d+(?:,\d{3})*(?:\.\d+)?(?:\s*(?:million|billion|trillion)\b)?"

    # 3. Percentages: Matches 50%, 12.5%, etc.
    percent_pattern = r"\b\d+(?:\.\d+)?%"

    # 4. General Numbers: Standalone digits, decimals, numbers with commas
    number_pattern = r"\b\d+(?:,\d{3})*(?:\.\d+)?\b"

    # Combine all patterns using the OR (|) operator
    combined_pattern = (
        f"{date_pattern}|{currency_pattern}|{percent_pattern}|{number_pattern}"
    )

    return re.sub(combined_pattern, "<XX>", text, flags=re.IGNORECASE)


In [11]:
for elem in old:
    print(mask(elem["text"]))

ITEM <XX>. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS
This report includes estimates, projections, statements relating to our business plans, objectives, and expected operating results that are “forward-looking statements” within the meaning of the Private Securities Litigation Reform Act of <XX>, Section 27A of the Securities Act of <XX>, and Section 21E of the Securities Exchange Act of <XX>. Forward-looking statements may appear throughout this report, including the following sections: “Management’s Discussion and Analysis of Financial Condition and Results of Operations” and “Risk Factors” (Part II, Item 1A of this Form <XX>-Q). These forward-looking statements generally are identified by the words “believe,” “project,” “expect,” “anticipate,” “estimate,” “intend,” “strategy,” “future,” “opportunity,” “plan,” “may,” “should,” “will,” “would,” “will be,” “will continue,” “will likely result,” and similar expressions. Forward-looking stateme

In [19]:
#  2. similarity_matrix: 2-D array of scores, shape (len(old_masked), len(new_masked)), 0-100
# scores[i, j]: how similar are paragraph i and j. This is done after both paragraphs are mask()ed
# we can use rapidfuzz for this

from rapidfuzz import process, fuzz

def similarity_matrix(old_masked, new_masked):

    matrix = process.cdist(old_masked, new_masked, scorer=fuzz.token_sort_ratio)

    return matrix


In [20]:
old_masked = []
mew_masked = []

for elem_old in old:
    old_masked_elem = mask(elem_old["text"])
    old_masked.append(old_masked_elem)
      
for elem_new in new:
    new_masked_elem = mask(elem_new["text"])    
    mew_masked.append(new_masked_elem)


In [21]:
print(similarity_matrix(old_masked, mew_masked))

[[100.          4.4582043  11.167513  ...   2.8921022   3.5135136
    2.8106508]
 [  4.4582043 100.         39.52334   ...  37.295437   37.078133
   44.972973 ]
 [ 11.418685   39.08046    98.669395  ...  39.22179    44.760212
   34.407364 ]
 ...
 [  3.5135136  37.078133   44.600525  ...  51.693157  100.
   47.05263  ]
 [  2.8106508  44.972973   34.494576  ...  47.11025    47.05263
  100.       ]
 [  3.194888   42.252327   42.750374  ...  49.331715   56.758575
   50.500237 ]]
